In [ ]:
# Notebook: Output and Impulse Response Computation with Detailed Technical Comment
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# 1. Setup Symbols and Matrices
s = sp.symbols('s')
t_sym = sp.symbols('t', real=True, positive=True)
y_func, h_func = sp.symbols('y_t h_t', cls=sp.Function)

A = sp.Matrix([[0, 2], [-1, -3]])
B = sp.Matrix([[0], [1]])
C = sp.Matrix([[2, 3]])
D = sp.Matrix([[-1]])
q0 = sp.Matrix([2, 1])
X_s = 0  

# 2. Compute Resolvent and Responses in Laplace Domain
Phi = (s * sp.eye(2) - A).inv()
Y_s = C * Phi * q0 + (C * Phi * B + D) * X_s
H_s = C * Phi * B + D  

# 3. Compute Inverse Laplace Transforms Symbolically
y_t_expr = sp.inverse_laplace_transform(Y_s[0, 0], s, t_sym)
h_t_expr = sp.inverse_laplace_transform(H_s[0, 0], s, t_sym)

print("--- Analytical Symbolic Expressions ---")
display(sp.Eq(y_func(t_sym), y_t_expr))
display(sp.Eq(h_func(t_sym), h_t_expr))

# 4. Technical Explanation Comment printed directly to the output
technical_comment = """
================================================================================
TECHNICAL COMMENT: THE DIRAC DELTA FUNCTION & JUPYTER / NUMERICAL LIMITATIONS
================================================================================
1. Mathematical Nature of the Dirac Delta:
   The impulse response h(t) includes a feedthrough term D = [-1], which transforms 
   in the time domain to the Dirac delta distribution, -delta(t). Mathematically, 
   the Dirac delta is a generalized function (distribution) rather than an ordinary 
   function; it is zero everywhere except at t = 0, where it is formally undefined 
   (tending to infinity) with an integral of 1.

2. Limitation in SymPy / Inverse Laplace Transform:
   While SymPy's symbolic engine can formally compute the inverse Laplace transform 
   involving constants (yielding terms like -DiracDelta(t)), standard plotting 
   backends and numerical conversion tools (such as NumPy / lambdify) cannot 
   evaluate a distribution numerically. A point-mass infinity cannot be sampled 
   on a discrete numerical array (np.linspace).

3. Jupyter Environment & Matplotlib Constraints:
   Jupyter executes code via underlying numerical and plotting libraries (like 
   Matplotlib). Matplotlib expects continuous or piecewise-continuous point arrays 
   y = f(t). Since delta(t) has infinite amplitude at a single infinitesimally thin 
   point (t=0) and zero elsewhere, standard line plots completely miss or omit it 
   unless handled explicitly as a discrete vertical stem line with a marker. 
   Consequently, only the continuous exponential components (e^{-t}u(t) + 2e^{-2t}u(t)) 
   are rendered in numerical plots.
================================================================================
"""
print(technical_comment)

# 5. Numerical Evaluation for Plotting (Continuous part of impulse response)
h_cont_expr = sp.exp(-t_sym) + 2 * sp.exp(-2 * t_sym)

f_y = sp.lambdify(t_sym, y_t_expr, 'numpy')
f_h_cont = sp.lambdify(t_sym, h_cont_expr, 'numpy')

t_vals = np.linspace(0.001, 5, 500)
y_vals = f_y(t_vals)
h_vals = f_h_cont(t_vals)

plt.figure(figsize=(8, 5))
plt.plot(t_vals, y_vals, label='$y(t)$ (System Output)', linewidth=2)
plt.plot(t_vals, h_vals, label='$h_{\\text{cont}}(t) = e^{-t} + 2e^{-2t}$ (Continuous part of $h(t)$)', linewidth=2, linestyle='--')
plt.title('System Output $y(t)$ and Continuous Impulse Response Components')
plt.xlabel('Time $t$ (s)')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.show()